# QIUP forward simulator — notebook demo

Frozen modular physics chain (5 modules).

Forward chain (source of truth):

1. **setup_params** — lab physics profile (`SetupParams` / YAML)
2. **latents** — operating point (`LatentState` / preset)
3. **optics** — object → SLM → SPDC → interference → optical PSF → `I±`
4. **detector** — EMCCD chain on ideal intensities
5. **simulate** — orchestrates optics+detector → frames + meta

Outputs are written under `examples/notebooks/_run_output/`.


## 0) Load the five modules (dependency order)


In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd()
if not (HERE / "setup_params.py").is_file():
    # Allow running the notebook from examples/notebooks/
    for candidate in (HERE.parent, HERE.parent.parent):
        if (candidate / "setup_params.py").is_file():
            HERE = candidate
            break
if not (HERE / "setup_params.py").is_file():
    raise FileNotFoundError("Run this notebook from the repository root (or examples/notebooks/).")

if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

OUT = HERE / "examples" / "notebooks" / "_run_output"
OUT.mkdir(parents=True, exist_ok=True)


def load_module(name: str, filename: str):
    path = HERE / filename
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


setup_params = load_module("setup_params", "setup_params.py")
latents = load_module("latents", "latents.py")
optics = load_module("optics", "optics.py")
detector = load_module("detector", "detector.py")
simulate = load_module("simulate", "simulate.py")

# Pillow for MNIST / image resize inside optics.load_object_M
try:
    from PIL import Image

    optics.Image = Image
except ImportError:
    Image = None

print("modules OK:")
for m in ("setup_params", "latents", "optics", "detector", "simulate"):
    print(" -", m)
print("output ->", OUT)


## 1) setup_params — lab physics profile


In [ ]:
YAML = HERE / "configs" / "setup_personal_template.yaml"
if not YAML.is_file():
    YAML = HERE / "setup_personal_template.yaml"

if YAML.is_file():
    setup = setup_params.load_setup_yaml(YAML)
    print("loaded YAML:", YAML)
else:
    setup = setup_params.SetupParams.personal_lab_template()
    print("YAML missing; using SetupParams.personal_lab_template()")

N = 512  # object / clean-phase image size (pixels)
N_REF = 64  # reference size for relative scaling (object/beam in frame)
GRID_SCALE = N / float(N_REF)

# Spatial scale proportional to N: at N_REF the physical beam waist fits correctly;
# when enlarging the canvas, scale waist and sigma_SPDC by the same ratio so the image stays in-frame.
fwd_ref = setup_params.resolve_forward(setup, N_REF)
setup = setup.overridden(
    use_legacy_beam_match=True,  # w_px âˆ N
    derive_sigma_spdc=False,
    sigma_spdc_px=max(0.5, float(fwd_ref.sigma_spdc_px) * GRID_SCALE),
    # Fabry–Pérot / etalon — tuned from lab Acquisition frames (diagonal FP bands).
    # Lab Michelson contrast ~0.18–0.22; we set a bit higher so dark/bright bands stand out more.
    enable_etalon_fringes=True,
    etalon_contrast=0.28,  # > lab (~0.2); stronger darkâ†”bright fringe contrast
    etalon_period_px=34.0,  # wider FP fringe bands (was 22)
    etalon_angle_deg=-40.0,  # diagonal bands like real Acquisition ROI
    etalon_phase_rad=0.0,
)
setup_params.save_setup_yaml(setup, OUT / "setup_used.yaml")
fwd = setup_params.resolve_forward(setup, N)
print(
    "grid scale    :",
    f"N={N} / N_REF={N_REF} â†’ Ã—{GRID_SCALE:.2f}",
    "| w_px=",
    round(float(fwd.w_px), 2),
    "| σ_SPDC=",
    round(float(fwd.sigma_spdc_px), 3),
)
print(
    "etalon FP     :",
    "ON" if setup.enable_etalon_fringes else "OFF",
    "| c=",
    setup.etalon_contrast,
    "| Î›_px=",
    round(float(setup.etalon_period_px), 2),
    "| Î±_deg=",
    setup.etalon_angle_deg,
)

print("setup_id      :", setup.setup_id)
print("model_version :", setup.model_version)
print("physics_tier  :", setup.physics_tier)
print("V0 / Î”L / L_c :", setup.V0, setup.delta_L_mm, setup.L_coh_mm)
print("pump_waist_um :", setup.pump_waist_um, "| pixel_um:", setup.pixel_um)
print("w_px / σ_SPDC :", round(fwd.w_px, 3), round(fwd.sigma_spdc_px, 3))
print("Î·_mode        :", round(fwd.eta_mode, 4))
print("two_beam / field_map:", setup.use_physical_two_beam, setup.use_field_overlap_visibility_map)


## 2) latents — operating point (preset + coupling)


In [ ]:
NUM_FRAMES = 16
PHOTON_COUNTING = False  # False â†’ Analog EMCCD (ADU); True â†’ photon-counting hit map
EM_GAIN = 250.0  # typical analog default (PC often uses ~800)
SEED = 42

state = latents.apply_preset(
    "well_aligned",
    num_frames=NUM_FRAMES,
    photon_counting=PHOTON_COUNTING,
    setup=setup,
    em_gain=EM_GAIN,
)

eff = state.effective()
print("preset        : well_aligned")
print("num_frames    :", state.num_frames)
print("photon_counting:", state.photon_counting)
print("em_gain       :", state.em_gain)
print("latents raw   :", {k: getattr(state, k) for k in latents.LATENT_KEYS})
print("latents eff   :", {k: round(eff[k], 4) for k in latents.LATENT_KEYS})

validity = latents.check_forbidden(state, fold_spdc_into_optical=False)
print("forbidden OK  :", validity.ok, "| violations:", validity.violations)


## 3) optics — clean reference (object / phase / ideal interference)


In [ ]:
ref_dir = OUT / "clean_ref"
OBJECT_BEAM_FILL = 0.85  # object support radius <= fill * w_px (inside 1/e^2 beam)


def make_synthetic_ref(
    N: int = 512,
    seed: int = 42,
    *,
    beam_waist_px: float,
    object_beam_fill: float = 0.85,
) -> dict:
    """Fallback without MNIST/Pillow — object inside beam profile."""
    max_side = int(max(8, min(N, round(object_beam_fill * np.sqrt(2.0) * beam_waist_px))))
    yy, xx = np.mgrid[0:N, 0:N].astype(np.float32)
    cx = cy = N / 2.0
    sig = max_side / 5.0
    M = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sig ** 2)).astype(np.float32)
    r2 = (xx - cx) ** 2 + (yy - cy) ** 2
    M = np.where(r2 <= (0.5 * max_side) ** 2, M, 0.0).astype(np.float32)
    M = optics.normalize(M)
    T = np.ones_like(M, dtype=np.float32)
    phi = (np.pi * M).astype(np.float32)
    D = (0.5 * M).astype(np.float32)
    return {
        "M": M,
        "T": T,
        "phi": phi,
        "D": D,
        "clean_D": D,
        "target_size": N,
        "seed": seed,
        "label": "synthetic_gaussian",
        "beam_waist_px": float(beam_waist_px),
        "object_beam_fill": float(object_beam_fill),
        "output_dir": str(ref_dir),
    }


try:
    ref = optics.build_clean_reference(
        output_dir=str(ref_dir),
        target_size=N,
        seed=SEED,
        mnist_index=2,
        beam_waist_px=float(fwd.w_px),
        object_beam_fill=OBJECT_BEAM_FILL,
    )
    print(
        "reference: MNIST | label=",
        ref.get("label"),
        "| w_px=",
        round(float(fwd.w_px), 3),
        "| object r<=",
        round(OBJECT_BEAM_FILL * float(fwd.w_px), 1),
        "px",
    )
except Exception as exc:
    print("build_clean_reference failed:", type(exc).__name__, exc)
    print("-> using synthetic phase object (beam-limited)")
    ref_dir.mkdir(parents=True, exist_ok=True)
    ref = make_synthetic_ref(
        N=N,
        seed=SEED,
        beam_waist_px=float(fwd.w_px),
        object_beam_fill=OBJECT_BEAM_FILL,
    )

yy, xx = np.mgrid[0:N, 0:N]
r = np.sqrt((xx - N / 2.0) ** 2 + (yy - N / 2.0) ** 2)
obj_mask = ref["M"] > 0.05 * float(np.max(ref["M"]))
obj_r_max = float(r[obj_mask].max()) if np.any(obj_mask) else 0.0
print("object r_max / beam w_px:", round(obj_r_max, 2), "/", round(float(fwd.w_px), 2))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(optics.normalize(ref["M"]), cmap="gray")
axes[0].set_title("Object M (<= beam)")
axes[1].imshow(ref["phi"], cmap="twilight")
axes[1].set_title("SLM phase phi")
axes[2].imshow(optics.normalize(ref["D"]), cmap="Blues_r")
axes[2].set_title("Clean D (ideal)")
env = np.exp(-2.0 * (r / max(float(fwd.w_px), 1e-6)) ** 2)
axes[3].imshow(optics.normalize(ref["M"]), cmap="gray")
axes[3].contour(env, levels=[np.exp(-2.0)], colors="cyan", linewidths=1.2)
axes[3].set_title("M + beam 1/e^2")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
fig.savefig(OUT / "01_optics_clean_reference.png", dpi=140, bbox_inches="tight")
plt.show()
plt.close(fig)


## 4) optics → detector — one frame, then analog EMCCD

Explicit two-stage view before the full `simulate` orchestration.
Camera output in **analog** mode = ADU frames (not a photon-counting hit map).


In [ ]:
rng = np.random.default_rng(SEED)

# --- optics: ideal intensities I+, I- ---
opt = optics._optical_intensity(ref, state, frame_idx=0, rng=rng)
I_plus = opt["I_plus"]
I_minus = opt["I_minus"]
vis = opt.get("vis") or {}

print("optical stages:", optics.OPTICAL_STAGES)
print("V_eff / V_coh / V_port:",
      round(float(vis.get("V_eff", float("nan"))), 4),
      round(float(vis.get("V_coh", float("nan"))), 4),
      round(float(vis.get("V_port", float("nan"))), 4))
print("w_px / sig_spdc / sig_opt:",
      opt.get("w_px"), opt.get("sig_spdc"), opt.get("sig_opt"))

# --- detector: analog EMCCD ADU frames ---
assert not state.photon_counting, "expected photon_counting=False for analog ADU"
prnu, hot = detector._fixed_patterns((N, N), state, rng)
Y_plus = detector._detector_pc_frame(I_plus, state, rng, prnu=prnu, hot=hot)
Y_minus = detector._detector_pc_frame(I_minus, state, rng, prnu=prnu, hot=hot)

stages = detector.DETECTOR_STAGES_ANALOG
print("detector mode : analog_emccd (ADU)")
print("detector stages:", stages)
print("mean I+ / mean ADU Y+:", float(I_plus.mean()), float(Y_plus.mean()))
print("ADU range Y+ / Y−:",
      (float(Y_plus.min()), float(Y_plus.max())),
      (float(Y_minus.min()), float(Y_minus.max())))

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
axes[0, 0].imshow(optics.normalize(I_plus), cmap="hot")
axes[0, 0].set_title("optics I+ (pre-detector)")
axes[0, 1].imshow(optics.normalize(I_minus), cmap="hot")
axes[0, 1].set_title("optics I− (pre-detector)")
axes[1, 0].imshow(optics.normalize(Y_plus), cmap="gray")
axes[1, 0].set_title("camera Y+ (analog ADU)")
axes[1, 1].imshow(optics.normalize(np.abs(Y_plus - Y_minus)), cmap="Blues_r")
axes[1, 1].set_title("|Y+ âˆ’ Y−| (analog)")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
fig.savefig(OUT / "02_optics_then_detector.png", dpi=140, bbox_inches="tight")
plt.show()
plt.close(fig)


## 5) simulate — full forward orchestration

`simulate_sequence` runs the optics→detector chain for all frames.


In [ ]:
result = simulate.simulate_sequence(
    ref,
    state,
    seed=SEED,
    raise_on_forbidden=False,
)

meta_vis = result.meta.get("vis") or {}
summary = {
    "setup_id": result.meta.get("setup_id"),
    "model_version": result.meta.get("model_version"),
    "physics_tier": result.meta.get("physics_tier"),
    "num_frames": result.meta.get("num_frames"),
    "photon_counting": result.meta.get("photon_counting"),
    "etalon_fringes": result.meta.get("etalon_fringes"),
    "em_gain_GEM": result.meta.get("em_gain_GEM"),
    "V_eff": float(meta_vis.get("V_eff", float("nan"))),
    "V_coh": float(meta_vis.get("V_coh", float("nan"))),
    "V_port": float(meta_vis.get("V_port", float("nan"))),
    "V0": float(meta_vis.get("V0", float("nan"))),
    "delta_L": float(meta_vis.get("delta_L", float("nan"))),
    "L_coh": float(meta_vis.get("L_coh", float("nan"))),
    "w_px": result.meta.get("w_px"),
    "sig_spdc": result.meta.get("sig_spdc"),
    "sig_opt": result.meta.get("sig_opt"),
    "mean_hits_plus": result.meta.get("mean_hits_plus"),
    "F2": result.meta.get("F2"),
    "latents_effective": result.meta.get("latents_effective"),
    "optical_stages": result.meta.get("optical_stages"),
    "detector_stages": result.meta.get("detector_stages"),
}

print(json.dumps(summary, indent=2, ensure_ascii=False, default=str))

np.savez_compressed(
    OUT / "frames.npz",
    frames_plus=result.frames_plus,
    frames_minus=result.frames_minus,
    frames_D=result.frames_D,
    clean_D=result.clean_D,
    sample=result.sample_dots,
)
(OUT / "meta_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
(OUT / "meta_full.json").write_text(
    json.dumps(result.meta, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

simulate.plot_clean_and_result(
    result,
    title=(
        f"Digital Twin | {summary['setup_id']} | "
        f"V_eff={summary['V_eff']:.3f} | N={summary['num_frames']}"
    ),
    save_path=str(OUT / "03_digital_twin_result.png"),
    show=True,
)


## 6) Saved files


In [ ]:
print("Saved under", OUT.resolve())
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size} bytes)")


## 7) Camera output — analog EMCCD (ADU)

Optics with near-ideal latents (~0), then the EMCCD chain in **analog** mode.
Camera output = quantized ADU frames, not a photon-counting hit map.


In [ ]:
# Ideal optics latents â†’ 0, then EMCCD analog ADU frames.
setup_ideal = setup.overridden(
    delta_M_rel=0.0,
    image_rotation_deg=0.0,
    wavefront_rms_waves=0.0,
    sigma_blur_extra_px=0.0,
    sigma_aberration_px=0.0,
)

state_an = latents.LatentState(
    s_pump=0.0,
    s_gain=0.0,
    s_align=0.0,
    s_thermal=0.0,
    s_det=0.0,
    s_cool=0.0,
    s_focus=0.0,
    s_slm=0.0,
    em_gain=float(EM_GAIN),
    exposure_s=float(state.exposure_s),
    photon_counting=False,
    num_frames=1,
    delta_phi_rms=0.0,
    setup=setup_ideal,
)

rng_an = np.random.default_rng(SEED)
opt_id = optics._optical_intensity(ref, state_an, frame_idx=0, rng=rng_an)
I_p = np.asarray(opt_id["I_plus"], dtype=np.float32)
I_m = np.asarray(opt_id["I_minus"], dtype=np.float32)
D_id = np.asarray(opt_id["D_ideal"], dtype=np.float32)

prnu_an, hot_an = detector._fixed_patterns((N, N), state_an, rng_an)
Y_p = detector._detector_pc_frame(I_p, state_an, rng_an, prnu=prnu_an, hot=hot_an)
Y_m = detector._detector_pc_frame(I_m, state_an, rng_an, prnu=prnu_an, hot=hot_an)
D_adu = (Y_p - Y_m).astype(np.float32)

print("=== Camera analog EMCCD (ADU) ===")
print("photon_counting:", state_an.photon_counting, "| em_gain:", state_an.em_gain)
print("detector stages:", detector.DETECTOR_STAGES_ANALOG)
print("mean I+ / mean ADU Y+:", float(I_p.mean()), float(Y_p.mean()))
print("ADU range Y+ / Y-:",
      (float(Y_p.min()), float(Y_p.max())),
      (float(Y_m.min()), float(Y_m.max())))

fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
axes[0, 0].imshow(optics.normalize(I_p), cmap="hot")
axes[0, 0].set_title("optics I+ (pre-camera)")
axes[0, 1].imshow(optics.normalize(I_m), cmap="hot")
axes[0, 1].set_title("optics I- (pre-camera)")
axes[0, 2].imshow(optics.normalize(D_id), cmap="RdBu_r")
axes[0, 2].set_title("optics D = I+ - I-")

axes[1, 0].imshow(optics.normalize(Y_p), cmap="gray")
axes[1, 0].set_title("camera Y+ (analog ADU)")
axes[1, 1].imshow(optics.normalize(Y_m), cmap="gray")
axes[1, 1].set_title("camera Y- (analog ADU)")
axes[1, 2].imshow(optics.normalize(D_adu), cmap="RdBu_r")
axes[1, 2].set_title("camera D = Y+ - Y-")

for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("EMCCD analog camera output (ADU)", fontsize=12)
plt.tight_layout()
fig.savefig(OUT / "04_camera_analog.png", dpi=140, bbox_inches="tight")
plt.show()
plt.close(fig)

np.savez_compressed(
    OUT / "camera_analog.npz",
    I_plus=I_p,
    I_minus=I_m,
    D_ideal=D_id,
    Y_plus_adu=Y_p,
    Y_minus_adu=Y_m,
    D_adu=D_adu,
)
print("saved ->", OUT / "04_camera_analog.png")
print("saved ->", OUT / "camera_analog.npz")
